<a href="https://colab.research.google.com/github/hongxu-yn/Drought-monitoring-and-assessment/blob/main/step00_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ==========================================
# 1. 环境准备与依赖安装
# ==========================================
!pip install -qU tqdm geemap earthengine-api

import ee
import geemap
import os
from google.colab import drive
from tqdm.auto import tqdm


drive.mount('/content/drive')

try:
    ee.Initialize(project='ee-hongxu')
except:
    ee.Authenticate()
    ee.Initialize(project='ee-hongxu')

roi = ee.Geometry.Rectangle([97, 20, 107, 30])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 数据下载-ndvi

In [11]:
def preprocess_ndvi(image):
    ndvi = image.select('NDVI')
    mask = ndvi.neq(-3000)
    ndvi_real = ndvi.updateMask(mask).multiply(0.0001)
    return ndvi_real.copyProperties(image, ['system:time_start', 'system:index'])

collection = ee.ImageCollection('MODIS/061/MOD13A3') \
    .filterBounds(roi) \
    .filterDate('2023-08-01', '2023-8-31') \
    .map(preprocess_ndvi)

tif_dir = '/content/drive/MyDrive/DATA/Satellite/MOD13A3_2001_2025'
os.makedirs(out_dir, exist_ok=True)

image_list = collection.toList(collection.size())
count = collection.size().getInfo()
print(f"\n🚀 准备下载 {count} 幅处理后的影像到: {out_dir}")

# 带进度条的循环下载
for i in tqdm(range(count), desc="下载进度", unit="月"):
    image = ee.Image(image_list.get(i))

    date = ee.Date(image.get('system:time_start')).format('YYYY_MM').getInfo()
    file_name = f'MODIS_NDVI_{date}.tif'
    out_path = os.path.join(tif_dir, file_name)

    if os.path.exists(out_path):
        continue

    geemap.ee_export_image(
        image,
        filename=out_path,
        scale=1000,
        region=roi,
        file_per_band=False
    )

print("\n✨ 所有影像下载完成！(无效值已被掩膜，有效值范围为 -0.2 到 1.0)")


🚀 准备下载 1 幅处理后的影像到: /content/drive/MyDrive/DATA/MOD13A3


下载进度:   0%|          | 0/1 [00:00<?, ?月/s]


✨ 所有影像下载完成！(无效值已被掩膜，有效值范围为 -0.2 到 1.0)


In [9]:
import os
import glob
import pandas as pd
import numpy as np
import xarray as xr
import gc

# 确保安装了 dask，它是 xarray 并行和内存优化的核心
try:
    import rioxarray
    import dask
except ImportError:
    !pip install rioxarray netCDF4 dask[array]
    import rioxarray

# 1. 设置输入输出路径
tif_dir = '/content/drive/MyDrive/DATA/Satellite/MOD13A3_2001_2025'
out_nc ='/content/drive/MyDrive/DATA/Satellite/ndvi_month_2001_2025.nc'

search_pattern = os.path.join(tif_dir, 'MODIS_NDVI_*.tif')
tif_files = glob.glob(search_pattern)
tif_files.sort()

if not tif_files:
    print(f"在 {tif_dir} 中未找到匹配 {search_pattern} 的文件，请检查！")
else:
    print(f"找到 {len(tif_files)} 个 NDVI 文件，启动 Dask 延迟加载模式...")

    data_arrays = []
    time_coords = []

    for fp in tif_files:
        basename = os.path.basename(fp)
        parts = basename.replace('.tif', '').split('_')

        try:
            year = int(parts[-2])
            month = int(parts[-1])
        except (IndexError, ValueError):
            continue

        time_obj = pd.Timestamp(year=year, month=month, day=1)
        time_coords.append(time_obj)

        # =========================================================
        # 核心优化 1：引入 chunks 参数，不把数据读入内存，而是建立虚拟映射
        # chunks={'x': 1000, 'y': 1000} 表示每次只处理 1000x1000 像素的块
        # =========================================================
        da = rioxarray.open_rasterio(fp, chunks={'x': 1000, 'y': 1000}).squeeze('band', drop=True)

        # 此时这些计算也是“虚拟”的，并没有真正消耗内存去计算
        da = da.astype('float32') * 0.0001
        da = da.where((da >= -1.0) & (da <= 1.0))

        data_arrays.append(da)

    print("\n构建时间拼接图 (未计算真实数据)...")
    time_index = pd.Index(time_coords, name='time')

    # 拼接 Dask 虚拟数组
    ds = xr.concat(data_arrays, dim=time_index)

    ds.name = 'ndvi'
    ds.attrs['description'] = 'Monthly NDVI (MOD13A3)'
    ds.attrs['units'] = '1'
    ds.rio.write_crs("EPSG:4326", inplace=True)

    if 'x' in ds.coords and 'y' in ds.coords:
        ds = ds.rename({'x': 'lon', 'y': 'lat'})
        ds.lon.attrs['units'] = 'degrees_east'
        ds.lat.attrs['units'] = 'degrees_north'

    print("\n开始执行分块计算并写入 NetCDF，这需要一些时间，但不会让内存崩溃...")

    # =========================================================
    # 核心优化 2：to_netcdf 会触发流式写入 (Streaming write)
    # 计算完一块就释放一块的内存
    # =========================================================
    ds.to_netcdf(out_nc, format='NETCDF4', engine='netcdf4', compute=True)

    # 清理内存垃圾
    del data_arrays, ds
    gc.collect()

    print(f"✨ 流式处理完成！文件已安全保存至: {out_nc}")

找到 299 个 NDVI 文件，启动 Dask 延迟加载模式...

构建时间拼接图 (未计算真实数据)...


/tmp/ipykernel_19905/1630470951.py:61: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'y' ('y',) The recommendation is to set join explicitly for this case.
  ds = xr.concat(data_arrays, dim=time_index)



开始执行分块计算并写入 NetCDF，这需要一些时间，但不会让内存崩溃...
✨ 流式处理完成！文件已安全保存至: /content/drive/MyDrive/DATA/Satellite/ndvi_month_2001_2025.nc


In [7]:
import os
import glob

# 设置您的文件夹路径
tif_dir = '/content/drive/MyDrive/DATA/Satellite/MOD13A3_2001_2025'
search_pattern = os.path.join(tif_dir, 'MODIS_NDVI_*.tif')
tif_files = glob.glob(search_pattern)

print(f"当前文件夹中实际找到文件数: {len(tif_files)}")

# 1. 提取实际存在的年月信息
actual_dates = set()
for fp in tif_files:
    basename = os.path.basename(fp)
    # 解析出 2001_01 这样的字符串
    parts = basename.replace('.tif', '').split('_')
    try:
        year = parts[-2]
        month = parts[-1]
        actual_dates.add(f"{year}_{month}")
    except (IndexError, ValueError):
        pass

# 2. 生成 2001-2025 年理论上应该存在的 300 个年月信息
expected_dates = set()
for year in range(2001, 2026):
    for month in range(1, 13):
        # 使用 :02d 确保月份是两位数，例如 01, 02
        expected_dates.add(f"{year}_{month:02d}")

# 3. 找不同（差集）
missing_dates = expected_dates - actual_dates

# 4. 打印结果
print("-" * 30)
if not missing_dates:
    print("太棒了！没有缺失月份，时间序列是完整的。")
else:
    print(f"⚠️ 发现缺失了 {len(missing_dates)} 个月！具体如下：")
    for date in sorted(list(missing_dates)):
        print(f" ❌ 缺失: MODIS_NDVI_{date}.tif")
print("-" * 30)

当前文件夹中实际找到文件数: 299
------------------------------
⚠️ 发现缺失了 1 个月！具体如下：
 ❌ 缺失: MODIS_NDVI_2013_08.tif
------------------------------


# 数据下载GPP

In [ ]:
import ee
import geemap

ee.Initialize()

start_year = 2001
end_year = 2025

years = ee.List.sequence(start_year, end_year)
months = ee.List.sequence(1, 12)

def process_gpp(image):
    # 修改点：将 select('GPP') 改为 select('Gpp')
    # 我们可以在重命名时将其改回全大写 rename('GPP')，这样后续代码就不需要改动了
    gpp = image.select('Gpp').multiply(0.0001).rename('GPP')

    return gpp.copyProperties(image, ['system:time_start'])

# 加载 MODIS GPP 8天产品集合
gpp_collection = (ee.ImageCollection("MODIS/061/MOD17A2H")
                  .filterBounds(roi)
                  .map(process_gpp))

# 5. 核心逻辑：逐年、逐月进行最大值合成 (MVC)
def by_year(y):
    def by_month(m):
        # 筛选出特定年月的所有 8 天影像
        filtered = gpp_collection.filter(ee.Filter.calendarRange(y, y, 'year')) \
                                 .filter(ee.Filter.calendarRange(m, m, 'month'))

        # 应用 max() 归约器提取最大值，并重新设置时间标签
        return filtered.max() \
                       .set('year', y) \
                       .set('month', m) \
                       .set('system:time_start', ee.Date.fromYMD(y, m, 1).millis()) \
                       .clip(roi) # 裁剪到研究区

    return months.map(by_month)

monthly_max_gpp = ee.ImageCollection.fromImages(years.map(by_year).flatten())

monthly_max_gpp = monthly_max_gpp.filter(ee.Filter.listContains('system:band_names', 'GPP'))

print("合成后的月度影像数量:", monthly_max_gpp.size().getInfo())

In [ ]:
import os
from tqdm.auto import tqdm  # 推荐使用 auto，在 Colab 或本地 Jupyter 中都能完美显示进度条

out_dir = '/content/drive/MyDrive/DATA/Satellite/MOD17A2HGF_2001_2025'
os.makedirs(out_dir, exist_ok=True)

# 2. 将前面生成的 ImageCollection 转换为 List，并获取总数
image_list = monthly_max_gpp.toList(monthly_max_gpp.size())
count = image_list.size().getInfo()

print(f"准备下载 {count} 景 GPP 月最大值影像到 '{out_dir}'...")

# 3. 使用 tqdm 循环批量下载
for i in tqdm(range(count), desc="下载进度", unit="月"):
    image = ee.Image(image_list.get(i))

    # 格式化日期：获取 system:time_start 并格式化为 YYYY_MM
    date = ee.Date(image.get('system:time_start')).format('YYYY_MM').getInfo()
    file_name = f'MOD17A2H_GPP_MVC_{date}.tif'
    out_path = os.path.join(out_dir, file_name)

    # 断点续传逻辑：如果文件已存在，则跳过
    if os.path.exists(out_path):
        continue

    try:
        # 直接导出到本地环境或挂载的云盘
        geemap.ee_export_image(
            image,
            filename=out_path,
            scale=500,  # MOD17A2H 产品的原生分辨率为 500 米
            region=roi,
            file_per_band=False # 我们只有一个 GPP 波段，设为 False 即可
        )
    except Exception as e:
        print(f"\n下载 {file_name} 时发生错误: {e}")

print("\n✨ 所有 GPP 影像下载完成！(原始值已乘以 0.0001，单位转化为 kg C/m²)")

# 逐月数据 合并为nc

In [2]:
import os
import glob
import pandas as pd
import xarray as xr

try:
    import rioxarray
except ImportError:
    !pip install rioxarray netCDF4
    import rioxarray

tif_dir = '/content/drive/MyDrive/DATA/Satellite/MOD17A2HGF_2001_2025'
out_nc = os.path.join(tif_dir, 'gpp_month_2001_2025.nc')

search_pattern = os.path.join(tif_dir, 'GPP_*.tif')
tif_files = glob.glob(search_pattern)
tif_files.sort()

if not tif_files:
    print(f"在 {tif_dir} 中未找到匹配 {search_pattern} 的文件，请检查！")
else:
    print(f"找到 {len(tif_files)} 个时间序列 TIF 文件，开始读取和时间堆叠...")

    data_arrays = []
    time_coords = []

    for fp in tif_files:
        basename = os.path.basename(fp)
        parts = basename.replace('.tif', '').split('_')

        try:
            year = int(parts[1])
            month = int(parts[2])
        except (IndexError, ValueError):
            continue

        time_obj = pd.Timestamp(year=year, month=month, day=1)
        time_coords.append(time_obj)

        da = rioxarray.open_rasterio(fp).squeeze('band', drop=True)
        data_arrays.append(da)

    time_index = pd.Index(time_coords, name='time')
    ds = xr.concat(data_arrays, dim=time_index)

    ds.name = 'gpp'
    ds.attrs['units'] = 'gC/m2'
    ds.rio.write_crs("EPSG:4326", inplace=True)

    if 'x' in ds.coords and 'y' in ds.coords:
        ds = ds.rename({'x': 'lon', 'y': 'lat'})
        ds.lon.attrs['units'] = 'degrees_east'
        ds.lat.attrs['units'] = 'degrees_north'

    ds.to_netcdf(out_nc, format='NETCDF4', engine='netcdf4')

    print(f"✨ 处理完成！多维时间序列已成功保存至: {out_nc}")

找到 300 个时间序列 TIF 文件，开始读取和时间堆叠...
✨ 处理完成！多维时间序列已成功保存至: /content/drive/MyDrive/DATA/Satellite/MOD17A2HGF_2001_2025/gpp_month_2001_2025.nc
